> **Generated notebook — do not edit here.**  
> Source: `01_scripts/01_quality_control.Rmd`, which is also a chapter of the course book.  
> To change anything, edit the Rmd and run `python3 util/rmd_to_ipynb.py`.
>
> Run the notebooks in order — **01 → 02 → 03** — with the **R** kernel; each step saves results that the next one loads.
>
> **Pick ONE dataset** (*S. aureus* or human) and work through it properly — if your group finishes early, start on the other one.

**First time here? Four things to expect:**

- 🌐 **Browser:** use **Chrome, Firefox or Edge** — Codespaces does not work reliably in Safari.
- 🧮 **Kernel:** when VS Code asks you to *Select Kernel*, choose **Jupyter Kernel...** → **R**. The notebooks run R, not Python.
- ⚠️ **"No text editor active" pop-up:** a harmless warning from the R extension — your code still runs. Just close it.
- ▶️ **Running cells:** use **Shift+Enter** or the ▶ button next to the cell — **not Ctrl+Enter**, which the R extension intercepts. The first cell can take a moment while the R kernel starts.

In [ ]:
# Match the report's defaults: warnings hidden (warning=FALSE in the Rmd)
# and 7 x 5 inch figures. Remove the warn option to see warnings.
options(warn = -1, repr.plot.width = 7, repr.plot.height = 5)

# Make tables display in Jupyter the way they do in the rendered report.
# kable()/kableExtra return HTML that the R kernel would otherwise show as
# raw text; DT::datatable() is an interactive widget whose JavaScript does
# not run in the VS Code output pane, so it is shown as a static table.
options(knitr.table.format = "html")
local({
  css <- paste0("<style>table.table,table.dataframe{border-collapse:collapse;font-size:0.9em}",
                ".table th,.table td{padding:3px 10px;border-bottom:1px solid #ddd}",
                ".table-striped tbody tr:nth-child(odd){background:#f5f7fa}</style>")
  registerS3method("repr_html", "knitr_kable", function(obj, ...) {
    paste0(css, paste(obj, collapse = "\n"))
  }, envir = asNamespace("repr"))
  registerS3method("repr_html", "datatables", function(obj, ...) {
    d <- as.data.frame(obj$x$data, stringsAsFactors = FALSE, check.names = FALSE)
    note <- if (nrow(d) > 100) sprintf(
      "<p style='font-size:0.85em;color:#666'><em>Static preview: first 100 of %d rows.</em></p>", nrow(d)) else ""
    tbl <- knitr::kable(head(d, 100), format = "html", row.names = FALSE,
                        table.attr = "class='table table-striped'")
    paste0(css, note, paste(tbl, collapse = "\n"))
  }, envir = asNamespace("repr"))
})

# Quality Control

This report covers the quality control and exploratory analysis of bulk RNA-seq data from primary human airway smooth muscle (ASM) cells treated with dexamethasone, from [Himes *et al.*, 2014](https://doi.org/10.1371/journal.pone.0099625) (GEO: GSE52778, PRJNA229998).

The list of contrasts of interest is:

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

- Dexamethasone (treatment) vs untreated (control)

</div>

**Topics covered**

-   Exploratory data analyses
-   Sanity Checks

**Data**

-   Reference genome: human GRCh37 (hg19), as used in the original paper.

-   Input data is the output from the [nf-core/rnaseq pipeline](https://nf-co.re/rnaseq/) `salmon.merged.gene.SummarizedExperiment.rds` (STAR + Salmon).

<div style="background:#eaf4fd;border-left:5px solid #3498db;border-radius:4px;padding:10px 16px;margin:10px 0;">

<strong>📘 Note:</strong> Key points about this dataset:
<ul>
<li>Reads are aligned with a splice-aware aligner (STAR), appropriate for spliced human transcripts</li>
<li>The reference GTF carries gene symbols, so genes are labelled directly (no ID conversion needed)</li>
<li>Enrichment analysis uses g:Profiler with human gene sets (GO, KEGG, Reactome)</li>
<li>Cells come from four donors → donor is a candidate confounder in the design (addressed below)</li>
</ul>

</div>

<div style="background:#eaf4fd;border-left:5px solid #3498db;border-radius:4px;padding:10px 16px;margin:10px 0;">

<strong>📘 Note on this experiment:</strong> The full dataset contains four treatment arms per donor
(untreated, albuterol, dexamethasone, albuterol+dexamethasone). Following the study's primary
comparison, this workshop uses only <strong>dexamethasone vs untreated</strong> (n = 4 per group).
Because each donor contributes one treated and one untreated sample, the design is <strong>paired</strong>.

</div>

**Explanation of the QC analysis 🧾**

This document guides you through the standard QC pipeline for bulk RNA-seq data processed with [nf-core/rnaseq](https://nf-co.re/rnaseq/) `STAR` + `Salmon`. It is structured to be both educational and reproducible.

## Setup the Environment

<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px;margin:10px 0;">

<strong>💡 Tip:</strong> Install packages only once!

</div>

<div style="background:#d1ecf1;border-left:4px solid #0c5460;padding:10px;margin:10px 0;">

<strong>📌 Remember:</strong> Load all libraries at the start of every session before running any analysis.

</div>

In [ ]:
library(tidyverse)
library(reshape2)
library(DESeq2)
library(ggpubr)
library(RColorBrewer)
library(pheatmap)
library(factoextra)
library(knitr)
library(kableExtra)
library(DT)

## Getting the Metadata

The metadata file describes the experimental design: which sample belongs to which donor and treatment group. This information is essential for `DESeq2` to model expression differences correctly.

In [ ]:
git_root <- system("git rev-parse --show-toplevel", intern = TRUE)

samples_info <- read.table(
  file.path(git_root, "data", "data-02-Homo_sapiens", "metadata", "metadata.tsv"),
  header      = TRUE,
  sep         = "\t",
  check.names = TRUE
)

<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px;margin:10px 0;">

<strong>💡 Tip:</strong> Use the interactive table below to verify your sample metadata before proceeding.

</div>

In [ ]:
DT::datatable(
  data       = samples_info,
  rownames   = FALSE,
  extensions = c('Buttons', 'Scroller'),
  options    = list(
    dom         = 'Bfrtip',
    buttons     = c('copy', 'csv'),
    deferRender = TRUE,
    scrollX     = TRUE,
    scrollY     = 200,
    scroller    = TRUE
  ),
  caption = 'Sample metadata — Human ASM (dexamethasone vs untreated)'
)

## Loading Count Data

The nf-core/rnaseq pipeline was run with `STAR` for alignment and `Salmon` for quantification. We load the `SummarizedExperiment` object produced by the pipeline, extract the raw count matrix, and assign gene symbols as row names.

<div style="background:#fdf2e9;border-left:5px solid #e67e22;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>⭐ Important:</strong> Salmon returns estimated counts, which are not integers. DESeq2 requires integer counts, so we round (not truncate) before building the object. This matches the tximport convention used by nf-core/differentialabundance.

</div>

<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px;margin:10px 0;">

<strong>💡 Tip:</strong> If you are unsure which assay name or rowData columns are available in your RDS, inspect them first with <code>assayNames(count_x)</code> and <code>names(rowData(count_x))</code>.

</div>

In [ ]:
count_x <- readRDS(
  file.path(git_root, "data", "data-02-Homo_sapiens", "hasapiens", "star_salmon",
            "salmon.merged.gene.SummarizedExperiment.rds")
)

# Inspect what is available (uncomment to explore):
# assayNames(count_x); names(rowData(count_x))

count_genes <- assay(count_x, assayNames(count_x)[1])

gene_symbols <- rowData(count_x)$gene_name
gene_ids     <- rowData(count_x)$gene_id

gene_symbols_saved <- ifelse(
  !is.na(gene_symbols) & nchar(gene_symbols) > 0,
  make.unique(as.character(gene_symbols)),
  make.unique(as.character(gene_ids))
)

# Salmon estimates are fractional -> round to nearest integer for DESeq2
count_genes <- as.matrix(count_genes)
mode(count_genes) <- "numeric"        # ensure numeric, not character
count_genes <- round(count_genes)
storage.mode(count_genes) <- "integer"
rownames(count_genes) <- gene_symbols_saved

cat("Dimensions (genes × samples):", dim(count_genes), "\n")
print(head(rownames(count_genes)))
print(colnames(count_genes))

# Report what the gene_id / gene_name columns actually contain.
# This pipeline used a RefSeq GTF with gtf_extra_attributes = gene_name,
# so both are expected to be symbols (not ENSEMBL). Confirm here:
cat("\nExample gene_id  :", paste(head(gene_ids, 3), collapse = ", "), "\n")
cat("Example gene_name:", paste(head(gene_symbols, 3), collapse = ", "), "\n")
cat("gene_id look like ENSEMBL? :",
    any(grepl("^ENSG", head(gene_ids, 50))), "\n")

<div style="background:#eaf4fd;border-left:5px solid #3498db;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>📘 Note:</strong> This dataset was processed against a <strong>RefSeq</strong> GRCh37 GTF with
  <code>gtf_extra_attributes = gene_name</code>, so both <code>gene_id</code> and <code>gene_name</code>
  hold gene <strong>symbols</strong> (e.g. <code>A1BG</code>), not ENSEMBL IDs. Symbols are exactly what
  g:Profiler and MSigDB accept, so no ID conversion is needed downstream. The check above prints the
  actual values so you can confirm this on your own RDS.

</div>

In [ ]:
# Persist gene_id <-> gene_name map for downstream scripts
id_map <- data.frame(
  gene_id     = as.character(gene_ids),
  gene_symbol = gene_symbols_saved,
  stringsAsFactors = FALSE
)
dir.create(file.path(git_root, "results", "human"), recursive = TRUE, showWarnings = FALSE)
saveRDS(id_map, file.path(git_root, "results", "human", "gene_id_symbol_map.rds"))

## Preparing the Data

Before building the `DESeq2` object we:

1.  Subset the count matrix to the samples in the metadata (dexamethasone + untreated only — the albuterol arms are excluded).
2.  Set factor levels so that `control` is the **reference level** (the denominator in fold-change calculations).
3.  Set `donor` as a factor (needed for the paired design).
4.  Align the sample order between the count matrix and the metadata — `DESeq2` requires these to match exactly.

In [ ]:
condition_levels   <- c("control", "treatment")
samples_info$group <- factor(samples_info$group, levels = condition_levels)
samples_info$donor <- factor(samples_info$donor)

# Subset count matrix to the 8 samples of interest (drop albuterol arms)
keep_samples <- intersect(samples_info$sample, colnames(count_genes))
missing      <- setdiff(samples_info$sample, colnames(count_genes))
if (length(missing) > 0) {
  stop("⛔ Metadata samples not found in count matrix: ", paste(missing, collapse = ", "))
}
count_genes <- count_genes[, keep_samples, drop = FALSE]

# Order metadata to match the count matrix columns exactly
samples_info <- samples_info[match(colnames(count_genes), samples_info$sample), ]

print(data.frame(count_col    = colnames(count_genes),
                 metadata_row = samples_info$sample,
                 donor        = samples_info$donor,
                 group        = samples_info$group))

## Creating the DESeqDataSet

The `DESeqDataSet` (DDS) holds the raw count matrix, sample metadata, and the design formula.

<div style="background:#eaf4fd;border-left:5px solid #3498db;border-radius:4px;padding:10px 16px;margin:10px 0;">

<strong>📘 Note:</strong> **Design choice.** Because each donor provides both a treated and an untreated
sample, donor is a paired/blocking factor. The recommended design is `~ donor + condition`, which
removes between-donor baseline differences before testing the treatment effect. We build the DDS with
this design, and also keep a simple `~ condition` version so the two can be compared for teaching (script 02).

</div>

In [ ]:
samples_info$condition <- factor(samples_info$group,
                                 levels = c("control", "treatment"))

# Primary (recommended) design: paired on donor
dds <- DESeqDataSetFromMatrix(
  countData = count_genes,
  colData   = samples_info,
  design    = ~ donor + condition
)

<div style="background:#eaf4fd;border-left:5px solid #3498db;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>📘 Note:</strong> The design formula uses R's formula syntax, where the tilde (`~`, pronounced **"TIL-duh"**)
means **"is modelled by"**. So `~ donor + condition` reads as: *"gene expression is modelled by
donor and condition"*.

In a general linear model, the tilde separates the **response variable** (left side) from
the **predictors** (right side). For example:

- `gene_expression ~ condition` — model expression as a function of condition only
- `gene_expression ~ donor + condition` — adjust for donor, then test condition

In DESeq2, the left side is omitted because the count matrix is already the response —
you only need to specify which variables explain the differences between samples.
The variable of interest (`condition`) goes **last**; the terms before it are adjusted for.

</div>

## Sanity Checks

<div style="background:#d1ecf1;border-left:4px solid #0c5460;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>📌 Remember:</strong> Always do a sanity check!

</div>

### Are We Working with Raw Counts?

`DESeq2` requires **raw, un-normalised integer counts**. Feeding it normalised values (TPM, FPKM) will produce incorrect results.

<div style="background:#d1ecf1;border-left:4px solid #0c5460;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>📌 Remember:</strong> Always verify your input before proceeding.

</div>

In [ ]:
options(scipen = 999)


kable(count_genes[1:6, ],
      caption = "Raw count matrix — first 6 genes",
      format.args = list(big.mark = ",")) %>%
  kable_styling(bootstrap_options = c("striped", "hover", "condensed"),
                full_width = FALSE)


barplot(colSums(count_genes),
        main   = "Library sizes (total counts per sample)",
        ylab   = "Total raw counts",
        xlab   = NULL,
        col    = "steelblue",
        las    = 2,
        names.arg = colnames(count_genes))

<div style="background:#fff3cd;border-left:4px solid #ffc107;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>💡 Tip:</strong> scipen = 999 is a penalty against scientific notation. R uses it to decide when to switch between fixed (150000) and scientific (1.5e+05) format. The default is scipen = 0 — by setting it to 999 you make the penalty so high that R almost never switches to scientific notation, preferring plain numbers instead.

</div>

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

Raw human RNA-seq counts span a very wide range (many genes near zero, a few in the hundreds of thousands). A highly right-skewed distribution is expected and correct at this stage.

</div>

### Pre-filtering Low-count Genes

Genes with very few counts across all samples carry no statistical power and inflate the multiple testing burden. We remove genes that do not have at least 10 counts in a minimum number of samples (equal to the size of the smallest group).

The human genome has roughly 20,000 protein-coding genes plus many non-coding ones, and a large fraction are not expressed in ASM cells — expect a substantial number to be filtered out.

In [ ]:
smallestGroupSize <- min(table(samples_info$condition))

cat("Smallest group size   :", smallestGroupSize, "\n")
cat("Filtering threshold   : at least 10 counts in", smallestGroupSize, "or more samples\n")

keep <- rowSums(counts(dds) >= 10) >= smallestGroupSize
n_before <- nrow(dds)
dds  <- dds[keep, ]

cat("Genes before filtering:", n_before, "\n")
cat("Genes after  filtering:", nrow(dds), "\n")
cat("Genes removed         :", sum(!keep), "\n")

### Factor Order and Reference Level

The first factor level is always the reference (denominator) in `DESeq2` comparisons. Setting it explicitly ensures fold changes are computed in the intended direction: **treatment (dexamethasone) vs control (untreated)**, not the reverse.

In [ ]:
dds$condition <- relevel(dds$condition, ref = "control")

levels(dds$condition)

<div style="background:#d4edda;border-left:4px solid #28a745;padding:10px;margin:10px 0;">

<strong>📌 Remember:</strong> The reference level determines the direction of fold changes. A positive log2FC means higher expression in dexamethasone-treated cells relative to untreated.

</div>

## Exploratory Data Analysis

In this section we do **not** perform statistical tests. The goal is **quality control:** check count distributions, detect technical outliers, and confirm that samples cluster as expected before committing to differential expression analysis.

### Estimate Size Factors

Library sizes (total mapped reads) differ between samples due to technical variation in sequencing depth. `DESeq2's` median-of-ratios normalisation corrects for this by computing a size factor per sample. Size factors close to 1.0 indicate balanced libraries; values far from 1.0 suggest uneven sequencing depth and warrant investigation.

In [ ]:
dds <- estimateSizeFactors(dds)

sizeFactors(dds) %>%
  enframe(name = "sample", value = "size_factor") %>%
  kable(digits = 3, caption = "DESeq2 size factors per sample") %>%
  kable_styling(bootstrap_options = c("striped", "hover"), full_width = FALSE)

### Distribution of Normalised Counts

Boxplots of log2-normalised counts per sample provide a quick visual check that all samples have comparable expression distributions. After normalisation, boxes should overlap substantially. A sample that is a clear outlier in median or spread may indicate a failed library or mislabelled sample.

<div style="background:#f8d7da;border-left:4px solid #721c24;padding:10px;margin:10px 0;">

<strong>📌 Remember:</strong> These normalised counts are for visualisation only. Always feed DESeq2 <strong>raw integer counts</strong>.

</div>

In [ ]:
normalized_counts <- counts(dds, normalized = TRUE)

counts_norm <- reshape2::melt(
  normalized_counts,
  varnames   = c("gene_id", "sample"),
  value.name = "counts"
)

counts_norm <- inner_join(
  counts_norm,
  as.data.frame(colData(dds)),
  by = "sample"
)

dir.create(file.path(git_root, "results", "human", "plots"), recursive = TRUE, showWarnings = FALSE)

distribution <- ggplot(counts_norm,
                       aes(x    = sample,
                           y    = log2(counts + 1),
                           fill = condition)) +
  geom_boxplot(outlier.size = 0.3, alpha = 0.8) +
  coord_flip() +
  theme_pubr(border = TRUE) +
  xlab("Sample") +
  ylab("log2(normalised counts + 1)") +
  ggtitle("Normalised count distribution — Human ASM")

distribution

ggsave(
  filename = file.path(git_root, "results", "human", "plots", "normalised_count_distribution.png"),
  plot     = distribution,
  width    = 8,
  height   = 6,
  dpi      = 300
)

## Sample Correlation Heatmap

Euclidean distances between `VST-transformed` samples reveal how similar samples are to each other globally. Samples from the same treatment group (or the same donor) may cluster together. With a paired design, donor structure is often visible here — this is expected and is exactly what the `~ donor + condition` model accounts for.

`Variance-stabilising transformation (VST)` is applied here because raw or normalised counts have heteroscedastic variance (high-count genes have much larger absolute variance).

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

VST removes this mean–variance dependence, making distances meaningful across the full expression range ([Anders & Huber, 2010](https://doi.org/10.1186/gb-2010-11-10-r106)).

</div>

In [ ]:
vsd <- varianceStabilizingTransformation(dds, blind = TRUE)

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

**`blind = TRUE`**, the VST is computed ignoring the experimental design. This is recommended for QC and exploratory analysis, where you want an unbiased view of sample similarity.

Use `blind = FALSE` only when the transformation is feeding into a model that already accounts for the design.

</div>

In [ ]:
sampleDists      <- dist(t(assay(vsd)))
sampleDistMatrix <- as.matrix(sampleDists)

rownames(sampleDistMatrix) <- paste(vsd$sample, vsd$condition, sep = " | ")
colnames(sampleDistMatrix) <- rownames(sampleDistMatrix)

heat <- pheatmap(
  sampleDistMatrix,
  main         = "Sample-to-sample distances (VST) — Human ASM",
  fontsize     = 10
)

heat

ggsave(
  filename = file.path(git_root, "results", "human", "plots", "heat.png"),
  plot     = heat,
  width    = 8,
  height   = 12,
  dpi      = 300
)

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

The diagonal is always zero (a sample compared to itself). Dark = similar, light = distant. Look at whether samples group by treatment, by donor, or both. If donor is a strong driver of the distances, that is a signal the paired design is worthwhile.

</div>

## Principal Component Analysis (PCA)

PCA reduces the high-dimensional expression space (one dimension per gene) to a small number of principal components that capture the largest sources of variance in the data. Inspect whether the treatment separates samples, and whether donor identity contributes to the remaining structure.

If a technical variable (batch, sequencing run, RNA quality) — or, here, donor — dominates PC1, that variance should be accounted for in the design before differential expression analysis ([Leek *et al.*, 2010](https://doi.org/10.1038/nrg2825)).

In [ ]:
pca_data <- plotPCA(vsd,
                    intgroup  = c("sample", "condition", "donor"),
                    returnData = TRUE)

pct_var <- round(100 * attr(pca_data, "percentVar"), 1)

PCAPlot <- ggplot(pca_data, aes(x     = PC1,
                                y     = PC2,
                                color = condition,
                                shape = donor,
                                label = sample)) +
  geom_point(size = 4) +
  geom_text(vjust = -0.8, size = 3, show.legend = FALSE) +
  geom_hline(yintercept = 0, linetype = "dashed", alpha = 0.3) +
  geom_vline(xintercept = 0, linetype = "dashed", alpha = 0.3) +
  theme_pubr(border = TRUE) +
  theme(
    axis.text       = element_text(size = 12),
    axis.title      = element_text(size = 14),
    legend.text     = element_text(size = 12),
    legend.position = "bottom"
  ) +
  labs(
    x     = paste0("PC1: ", pct_var[1], "% variance"),
    y     = paste0("PC2: ", pct_var[2], "% variance"),
    title = "PCA — Human ASM (VST-transformed counts)",
    color = "Condition",
    shape = "Donor"
  )

PCAPlot

ggsave(
  filename = file.path(git_root, "results", "human", "plots", "PCA_Plot.png"),
  plot     = PCAPlot,
  width    = 8,
  height   = 6,
  dpi      = 300
)

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

Colour encodes treatment, shape encodes donor. If treatment separates cleanly on one axis while donors are spread along another, that is the classic signature of a paired design: a consistent treatment effect on top of donor-to-donor baseline differences.

</div>

### 🧠 Interpretation questions

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

1. Find each donor's treated and untreated points in your PCA. Which is bigger: the difference between donors, or the shift caused by treatment? Based on that, predict which model will find more differentially expressed genes in the next notebook — `~ condition` or `~ donor + condition` — and why. Write the prediction down: you will check it.

</div>

## Detecting Outliers

The PCA above uses only the top 500 most variable genes (DESeq2 default). Here we run PCA on the full VST matrix and inspect a scree plot and biplot to assess whether any single sample drives an unusual amount of variance, a common sign of a technical outlier.

In [ ]:
pca_full <- prcomp(t(assay(vsd)))

screeplot <- fviz_screeplot(pca_full, addlabels = TRUE,
               main = "Scree plot — variance per PC")
screeplot
pca_ind <- fviz_pca_ind(pca_full, geom = c("point", "text"), repel = TRUE,
             title = "PCA — sample positions (full gene matrix)")
pca_ind
pca_biplot <- fviz_pca_biplot(pca_full,
                repel        = TRUE,
                select.var   = list(contrib = 50),  # top genes only
                title        = "Biplot — top 50 contributing genes and samples")

pca_biplot

ggsave(
  filename = file.path(git_root, "results", "human", "plots", "pca_screeplot.png"),
  plot     = screeplot,
  width    = 8,
  height   = 6,
  dpi      = 300
)

ggsave(
  filename = file.path(git_root, "results", "human", "plots", "pca_individuals.png"),
  plot     = pca_ind,
  width    = 8,
  height   = 6,
  dpi      = 300
)

ggsave(
  filename = file.path(git_root, "results", "human", "plots", "pca_biplot.png"),
  plot     = pca_biplot,
  width    = 8,
  height   = 6,
  dpi      = 300
)

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

**How to read a biplot:**

**Dots** = samples

**Arrows/lines** = genes — the direction shows which samples that gene is highly expressed in, and the length shows how strongly it contributes to the PC:

- Genes pointing **right** → higher expression in samples on the right
- Genes pointing **left** → higher expression in samples on the left
- Genes near the **centre** → contribute little to either PC

**What to look for:**

- Genes with long arrows are the strongest candidates for driving the dominant axis of variation
- If many arrows point in the same direction, it suggests coordinated regulation (a pathway-level response)
- Check whether the dominant axis aligns with treatment or with donor

</div>

## Top Variable Genes Heatmap

Clustering the 50 most variable genes across samples provides a gene-level view of the structure in the data. Truly treatment-responsive genes should show block structure by condition. This heatmap also helps reveal whether donor identity contributes strongly to the top variable genes.

Row-scaling (z-score per gene) is applied so that highly expressed genes do not visually dominate over lowly expressed ones.

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 12)  # figure size set in the Rmd chunk
topVarGenes <- head(order(rowVars(assay(vsd)), decreasing = TRUE), 50)

df_anno <- as.data.frame(colData(vsd)[, c("condition", "donor"), drop = FALSE])

heat_plot <- pheatmap(
  assay(vsd)[topVarGenes, ],
  cluster_cols             = TRUE,
  cluster_rows             = TRUE,
  scale                    = "row",
  clustering_distance_rows = "euclidean",
  clustering_distance_cols = "euclidean",
  annotation_col           = df_anno,
  show_colnames            = TRUE,
  show_rownames            = TRUE,
  fontsize_row             = 7,
  main                     = "Top 50 variable genes — Human ASM (VST, row-scaled)"
)

heat_plot

png(
  filename = file.path(git_root, "results", "human", "plots", "heatmap_top50_variable.png"),
  width    = 8,
  height   = 12,
  units    = "in",
  res      = 300
)
heat_plot
dev.off()

<div style="background:#eaf4fd;border-left:5px solid #3498db;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>📘 Note:</strong> Rows are already gene symbols (from the human reference), so they are ready
  to use directly in the enrichment analysis (script 03) with no ID conversion step.

</div>

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

**How to read this heatmap:**

- Each **row** is a gene, each **column** is a sample
- Colours show the **z-score** (row-scaled expression) — red = higher than average for that gene, blue = lower
- The **dendrograms** show hierarchical clustering — samples/genes that behave similarly are grouped together

**What to look for:**

- Clear colour blocks by condition → the treatment has a strong, consistent transcriptional effect
- Whether samples cluster by donor instead of (or in addition to) treatment — a visual argument for the paired design

</div>

### 🧠 Interpretation questions

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

2. Pick the gene with the cleanest treated/untreated split in your heatmap. Which of these can you claim from it: more transcript? more protein? more activity? Then look for *NR3C1* — the receptor dexamethasone binds to. It is a pre-existing protein: did its transcript need to change for the response you see? What does that say about what RNA-seq can and cannot show?

</div>

## Summary

Before proceeding to differential expression analysis, confirm the QC checks and note the design implication:

<div style="background:#f0f4f8;border:1px solid #c9d6e3;border-radius:4px;padding:10px 20px;margin:10px 0;">

| Check | What to confirm |
|---|---|
| Size factors ≈ 1.0 across samples | Library sizes are balanced |
| Boxplots of normalised counts overlap | No extreme outlier samples |
| Correlation heatmap | Note whether structure is by treatment, by donor, or both |
| PCA | Note whether treatment or donor drives the main axes |
| No isolated samples | No technical outliers |

</div>

<div style="font-size: 0.9em; color: grey;">

*Because the four donors each contribute a treated and an untreated sample, donor is a paired factor.
The recommended design `~ donor + condition` removes donor baseline differences before testing the
treatment effect. Script 02 runs this model and, for teaching, compares it against the simpler
`~ condition` model.*

</div>

<div style="background:#fdf2e9;border-left:5px solid #e67e22;border-radius:4px;padding:10px 16px;margin:10px 0;">

  <strong>⭐ Important:</strong> If any check fails, investigate the cause before running DESeq2. Proceeding with outlier samples or a mis-specified design will compromise all downstream results.

</div>

In [ ]:
# ── Export DDS for downstream analysis ────────────────────────────────────────
results_dir <- file.path(git_root, "results", "human", "rds")
dir.create(results_dir, recursive = TRUE, showWarnings = FALSE)

dds_path <- file.path(results_dir, "dds_human_asm.rds")
saveRDS(dds, file = dds_path)

```r
cat("✅ DDS saved to:", dds_path, "\n")
cat("   Dimensions  :", nrow(dds), "genes ×", ncol(dds), "samples\n")
cat("   Design      : ~ donor + condition\n")
cat("   Conditions  :", paste(levels(dds$condition), collapse = " vs "), "\n")
```

</br>

```r
sessionInfo()
```